In [ ]:
!pip install deep-translator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 2.5 MB/s eta 0:00:00


In [ ]:
import json
import re
import unicodedata

input_file = "annotations.json"
output_file = "annotations_corrected.json"

# ------------------- Fonctions utilitaires -------------------

def normalize_text(text):
    """Supprime les accents et met en minuscule"""
    text_nfkd = unicodedata.normalize('NFKD', text)
    return ''.join([c for c in text_nfkd if not unicodedata.combining(c)]).lower()

def find_all_occurrences_super_robust(text, substring):
    """
    Recherche très robuste : insensible à la casse, accents, espaces multiples et caractères spéciaux.
    Retourne toutes les positions (start, end) du substring dans text.
    """
    norm_text = normalize_text(text)
    norm_sub = normalize_text(substring)

    # Remplacer les espaces multiples par un espace unique
    norm_text = re.sub(r'\s+', ' ', norm_text)
    norm_sub = re.sub(r'\s+', ' ', norm_sub)

    positions = []
    start_search = 0
    while start_search < len(norm_text):
        index = norm_text.find(norm_sub, start_search)
        if index == -1:
            break
        # Retrouver la position exacte dans le texte original
        pattern = re.escape(substring)
        match = re.search(pattern, text[start_search:], re.IGNORECASE)
        if match:
            real_start = start_search + match.start()
            real_end = start_search + match.end()
            positions.append((real_start, real_end))
            start_search = real_end
        else:
            start_search += len(substring)
    return positions

def is_position_valid_strict(pos, used_positions):
    """
    Vérifie que :
    - start et end sont uniques (pas de duplication)
    - pas de chevauchement avec d'autres positions Actor
    """
    start, end = pos
    for u_start, u_end, _ in used_positions:
        # Duplication totale interdite
        if start == u_start and end == u_end:
            return False
        # Duplication partielle interdite
        if start == u_start or end == u_end:
            return False
        # Chevauchement interdit
        if start < u_end and end > u_start:
            return False
    return True

# ------------------- Traitement -------------------

with open(input_file, 'r', encoding='utf-8') as f:
    data = json.load(f)

for entry in data["data"]:
    input_text = entry["input"]
    used_actor_positions = []  # stocke (start, end, text) pour Actor

    for ann in entry["annotations"]:
        label = ann["label"]
        ann_text = ann["text"]
        occurrences = find_all_occurrences_super_robust(input_text, ann_text)
        nb_occurrences = len(occurrences)

        if nb_occurrences == 0:
            print(f"[NOT FOUND] Texte '{ann_text}' non trouvé dans l'input ID {entry['ID']}. Occurrences trouvées = 0, Annotations manquantes = 1")
            continue

        if label == "Actor":
            # Chercher occurrence valide strict
            valid_occurrence = None
            for occ in occurrences:
                if is_position_valid_strict(occ, used_actor_positions):
                    valid_occurrence = occ
                    break

            if valid_occurrence is None:
                nb_missing = 1
                print(f"[MISSING] Actor '{ann_text}' ID {entry['ID']}: Occurrences trouvées = {nb_occurrences}, Annotations manquantes = {nb_missing}")
                continue

            start, end = valid_occurrence
            used_actor_positions.append((start, end, ann_text))
        else:
            # Autres labels → première occurrence
            start, end = occurrences[0]

        ann["start"] = start
        ann["end"] = end

# ------------------- Sauvegarde -------------------

with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print(f"✅ Positions corrigées strictes pour Actor sauvegardées dans {output_file}")


Traduction terminée ! Résultat enregistré dans translated_codeseqnece.json


In [ ]:
!pip install tqdm


In [ ]:
def is_position_valid_smart(pos, used_positions, ann_text, input_text):
    """
    Vérifie si une position peut être utilisée :
    - Aucun chevauchement strict avec d'autres annotations Actor
    - Permet des sous-mots (ex: 'vendeur' dans 'vendeurs') si pas encore utilisé
    """
    start, end = pos

    for u_start, u_end, u_text in used_positions:
        # Exact overlap interdit
        if start == u_start and end == u_end:
            return False
        # chevauchement strict interdit
        if start < u_end and end > u_start:
            # autoriser si le texte est différent et est inclus dans l'autre
            used_norm = normalize_text(u_text)
            ann_norm = normalize_text(ann_text)
            if ann_norm in used_norm or used_norm in ann_norm:
                continue
            return False
    return True


✅ Positions corrigées (texte exact + variantes singulier/pluriel sans itertools) sauvegardées dans annotations_corrected.json


In [ ]:
import json
from collections import defaultdict

# Charger le fichier JSON
input_file = "/content/annotations_corrected.json"  # le fichier déjà corrigé
with open(input_file, 'r', encoding='utf-8') as f:
    data = json.load(f)

for entry in data["data"]:
    # Dictionnaire pour tracker les positions utilisées pour Actor
    position_map = defaultdict(list)

    for ann in entry["annotations"]:
        if ann["label"] == "Actor":
            pos = (ann["start"], ann["end"])
            position_map[pos].append(ann["id"])

    # Vérifier les positions partagées
    for pos, ids in position_map.items():
        if len(ids) > 1:
            print(f"[CONFLIT] ID {entry['ID']}: Les Actors {ids} partagent la même position {pos}")
